In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🔍 EDA Complète — Hackathon SGDB 2026
# MAGIC Tous les résultats sont collectés et affichés dans UN SEUL print à la fin.

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings("ignore")

# Accumulateur de résultats
R = []

def section(title):
    R.append(f"\n{'='*80}")
    R.append(f"  {title}")
    R.append(f"{'='*80}\n")

def line(text=""):
    R.append(str(text))

def df_to_str(pdf, max_rows=30):
    return pdf.head(max_rows).to_string(index=False)

# COMMAND ----------

# Chargement
train    = spark.table("workspace.default.histo_ventes_train")
test     = spark.table("workspace.default.histo_ventes_test")
agences  = spark.table("workspace.default.donnees_agence")
articles = spark.table("workspace.default.donnees_articles")
factu    = spark.table("workspace.default.donnees_facturation")

n_train = train.count()
n_test  = test.count()
train_cols = set(train.columns)

print("Tables chargées ✅  — Analyse en cours, patience...")

# COMMAND ----------

# 1. VUE D'ENSEMBLE
section("1. VUE D'ENSEMBLE")

for name, df in [("histo_ventes_train", train), ("histo_ventes_test", test),
                  ("donnees_agence", agences), ("donnees_articles", articles),
                  ("donnees_facturation", factu)]:
    total = df.count()
    line(f"\n--- {name} --- ({total:,} lignes, {len(df.columns)} colonnes)")
    stats = []
    for c in df.columns:
        cond = F.col(c).isNull()
        if str(df.schema[c].dataType) == "StringType":
            cond = cond | (F.col(c) == "")
        nulls = df.where(cond).count()
        distinct = df.select(c).distinct().count()
        stats.append({"colonne": c, "type": str(df.schema[c].dataType),
                       "nulls": nulls, "null_%": round(nulls/total*100, 2), "distinct": distinct})
    line(df_to_str(pd.DataFrame(stats)))

# COMMAND ----------

# 2. SAMPLES
section("2. SAMPLE DE CHAQUE TABLE (5 lignes)")
for name, df in [("histo_ventes_train", train), ("histo_ventes_test", test),
                  ("donnees_agence", agences), ("donnees_articles", articles),
                  ("donnees_facturation", factu)]:
    line(f"\n--- {name} ---")
    line(df_to_str(df.limit(5).toPandas()))

# COMMAND ----------

# 3. STATS QUANTITE
section("3. STATISTIQUES QUANTITE")
pdf_q = train.select("quantite").toPandas()
pct_zero = (pdf_q["quantite"] == 0).mean() * 100
line(f"  Lignes totales  : {len(pdf_q):,}")
line(f"  % de zéros      : {pct_zero:.1f}%")
line(f"  Min             : {pdf_q['quantite'].min()}")
line(f"  Médiane         : {pdf_q['quantite'].median()}")
line(f"  Moyenne         : {pdf_q['quantite'].mean():.2f}")
line(f"  P75             : {pdf_q['quantite'].quantile(0.75)}")
line(f"  P90             : {pdf_q['quantite'].quantile(0.90)}")
line(f"  P95             : {pdf_q['quantite'].quantile(0.95)}")
line(f"  P99             : {pdf_q['quantite'].quantile(0.99)}")
line(f"  Max             : {pdf_q['quantite'].max()}")
line(f"  Ecart-type      : {pdf_q['quantite'].std():.2f}")

# COMMAND ----------

# 4. TEMPOREL
section("4. ANALYSE TEMPORELLE")
sem_train = train.select("semaine").distinct().orderBy("semaine").toPandas()
sem_test  = test.select("semaine").distinct().orderBy("semaine").toPandas()
line(f"  Semaines train : {sem_train['semaine'].iloc[0]} -> {sem_train['semaine'].iloc[-1]} ({len(sem_train)} sem)")
line(f"  Semaines test  : {sem_test['semaine'].iloc[0]} -> {sem_test['semaine'].iloc[-1]} ({len(sem_test)} sem)")

weekly = (
    train.groupBy("semaine")
    .agg(F.sum("quantite").alias("total_qty"),
         F.round(F.avg("quantite"), 3).alias("avg_qty"),
         F.count("*").alias("nb_lignes"),
         F.sum(F.when(F.col("quantite")==0,1).otherwise(0)).alias("nb_zeros"))
    .withColumn("pct_zero", F.round(F.col("nb_zeros")/F.col("nb_lignes")*100, 1))
    .orderBy("semaine").toPandas()
)
line(f"\n  Top 10 semaines volume :")
line(df_to_str(weekly.nlargest(10, "total_qty")[["semaine","total_qty","avg_qty","pct_zero"]]))
line(f"\n  Bottom 10 semaines volume :")
line(df_to_str(weekly.nsmallest(10, "total_qty")[["semaine","total_qty","avg_qty","pct_zero"]]))

# COMMAND ----------

# 5. SAISONNALITE
section("5. PROFIL SAISONNIER")
train_sem = train.withColumn("annee", F.split("semaine","-")[0].cast("int")) \
                 .withColumn("num_semaine", F.split("semaine","-")[1].cast("int"))

season = (
    train_sem.groupBy("num_semaine")
    .agg(F.round(F.avg("quantite"),3).alias("avg_qty"), F.sum("quantite").alias("total_qty"))
    .orderBy("num_semaine").toPandas()
)
line("  Profil saisonnier (avg_qty par num_semaine) :")
line(df_to_str(season, 53))

season_yr = (
    train_sem.groupBy("annee","num_semaine")
    .agg(F.round(F.avg("quantite"),3).alias("avg_qty"))
    .orderBy("annee","num_semaine").toPandas()
)
line(f"\n  Années : {sorted(season_yr['annee'].unique())}")
for yr in sorted(season_yr["annee"].unique()):
    sub = season_yr[season_yr["annee"]==yr]
    line(f"    {yr} : min={sub['avg_qty'].min():.3f}  max={sub['avg_qty'].max():.3f}  mean={sub['avg_qty'].mean():.3f}")

# COMMAND ----------

# 6. AGENCES
section("6. STATS PAR AGENCE")
agence_stats = (
    train.groupBy("code_agence")
    .agg(F.sum("quantite").alias("total_qty"),
         F.round(F.avg("quantite"),3).alias("avg_qty"),
         F.countDistinct("code_article").alias("nb_articles"),
         F.countDistinct("semaine").alias("nb_semaines"),
         F.round(F.sum(F.when(F.col("quantite")==0,1).otherwise(0))/F.count("*")*100,1).alias("pct_zero"))
    .orderBy(F.desc("total_qty")).toPandas()
)
line(f"  Nb agences : {len(agence_stats)}")
line(f"\n  Top 20 agences :")
line(df_to_str(agence_stats.head(20)))
line(f"\n  Agences >90% zéros : {(agence_stats['pct_zero']>90).sum()}")
line(f"  Agences >50% zéros : {(agence_stats['pct_zero']>50).sum()}")

# COMMAND ----------

# 7. ENRICHISSEMENT AGENCES
section("7. ENRICHISSEMENT AGENCES")
ag_cols = agences.columns
line(f"  Colonnes : {ag_cols}")
for col_name in ag_cols:
    if col_name not in train_cols and col_name != "code_agence":
        line(f"\n  --- Répartition par {col_name} ---")
        res = (
            train.join(agences.select("code_agence", col_name), "code_agence", "left")
            .groupBy(col_name)
            .agg(F.sum("quantite").alias("total_qty"),
                 F.round(F.avg("quantite"),3).alias("avg_qty"),
                 F.countDistinct("code_agence").alias("nb_agences"))
            .orderBy(F.desc("total_qty")).limit(30).toPandas()
        )
        line(df_to_str(res))

# COMMAND ----------

# 8. ARTICLES
section("8. STATS PAR ARTICLE")
article_stats = (
    train.groupBy("code_article")
    .agg(F.sum("quantite").alias("total_qty"),
         F.round(F.avg("quantite"),3).alias("avg_qty"),
         F.countDistinct("code_agence").alias("nb_agences"),
         F.countDistinct("semaine").alias("nb_semaines"),
         F.round(F.sum(F.when(F.col("quantite")==0,1).otherwise(0))/F.count("*")*100,1).alias("pct_zero"))
    .orderBy(F.desc("total_qty")).toPandas()
)
line(f"  Nb articles : {len(article_stats)}")
line(f"\n  Top 20 articles :")
line(df_to_str(article_stats.head(20)))
art_sorted = article_stats.sort_values("total_qty", ascending=False)
art_sorted["cum_pct"] = art_sorted["total_qty"].cumsum() / art_sorted["total_qty"].sum() * 100
n80 = (art_sorted["cum_pct"]<=80).sum()
line(f"\n  Pareto : {n80} articles ({n80/len(article_stats)*100:.1f}%) = 80% du volume")
line(f"  Articles >90% zéros : {(article_stats['pct_zero']>90).sum()}")

# COMMAND ----------

# 9. ENRICHISSEMENT ARTICLES
section("9. ENRICHISSEMENT ARTICLES")
art_cols = articles.columns
line(f"  Colonnes : {art_cols}")
for col_name in art_cols:
    if col_name not in train_cols and col_name != "code_article":
        line(f"\n  --- Répartition par {col_name} ---")
        res = (
            train.join(articles.select("code_article", col_name), "code_article", "left")
            .groupBy(col_name)
            .agg(F.sum("quantite").alias("total_qty"),
                 F.round(F.avg("quantite"),3).alias("avg_qty"),
                 F.countDistinct("code_article").alias("nb_articles"))
            .orderBy(F.desc("total_qty")).limit(30).toPandas()
        )
        line(df_to_str(res))

# COMMAND ----------

# 10. FACTURATION
section("10. FACTURATION")
fac_cols = factu.columns
line(f"  Colonnes : {fac_cols}")
line(f"\n  Sample (5 lignes) :")
line(df_to_str(factu.limit(5).toPandas()))
line(f"\n  Stats summary :")
line(df_to_str(factu.summary().toPandas()))

# COMMAND ----------

# 11. PAIRES
section("11. PAIRES AGENCE x ARTICLE")
nb_pair_train = train.select("code_agence","code_article").distinct().count()
nb_pair_test  = test.select("code_agence","code_article").distinct().count()

paires_test_only = (
    test.select("code_agence","code_article").distinct()
    .join(train.select("code_agence","code_article").distinct(),
          ["code_agence","code_article"], "left_anti").count()
)
ag_test_only = (
    test.select("code_agence").distinct()
    .join(train.select("code_agence").distinct(), "code_agence", "left_anti").count()
)
art_test_only = (
    test.select("code_article").distinct()
    .join(train.select("code_article").distinct(), "code_article", "left_anti").count()
)

line(f"  Paires uniques train         : {nb_pair_train:,}")
line(f"  Paires uniques test          : {nb_pair_test:,}")
line(f"  Paires test ABSENTES train   : {paires_test_only:,} ({paires_test_only/nb_pair_test*100:.1f}%)")
line(f"  Agences test absentes train  : {ag_test_only}")
line(f"  Articles test absents train  : {art_test_only}")

pair_stats = (
    train.groupBy("code_agence","code_article")
    .agg(F.sum("quantite").alias("total_qty"),
         F.round(F.avg("quantite"),3).alias("avg_qty"),
         F.count("*").alias("nb_semaines"),
         F.round(F.sum(F.when(F.col("quantite")==0,1).otherwise(0))/F.count("*")*100,1).alias("pct_zero"))
    .toPandas()
)
line(f"\n  Distribution paires :")
line(f"    100% zéros     : {(pair_stats['pct_zero']==100).sum():,}")
line(f"    >90% zéros     : {(pair_stats['pct_zero']>90).sum():,}")
line(f"    avg_qty > 10   : {(pair_stats['avg_qty']>10).sum():,}")
line(f"    avg_qty > 100  : {(pair_stats['avg_qty']>100).sum():,}")
line(f"    1 seule sem    : {(pair_stats['nb_semaines']==1).sum():,}")

# COMMAND ----------

# 12. CORRELATIONS LAGS
section("12. CORRELATIONS LAGS")
w = Window.partitionBy("code_agence","code_article").orderBy("semaine")

for lag_n in [1, 4, 12, 26, 52]:
    train_lag_n = (
        train
        .withColumn(f"lag", F.lag("quantite", lag_n).over(w))
        .where(F.col("lag").isNotNull())
    )
    pdf_tmp = train_lag_n.select("quantite", "lag").sample(0.1).toPandas()
    corr_n = pdf_tmp["quantite"].corr(pdf_tmp["lag"])
    n_avail = train_lag_n.count()
    line(f"  Lag-{lag_n:>2} : corr = {corr_n:.4f}  (n={n_avail:,})")

# COMMAND ----------

# 13. TABLEAU FEATURES
section("13. TABLEAU RECAPITULATIF FEATURES")
features_table = pd.DataFrame([
    {"Feature":"qty_lag52","Cat":"Lag","Desc":"Même semaine N-1","Prio":"*****","Danger":"NaN si paire nouvelle"},
    {"Feature":"qty_lag1","Cat":"Lag","Desc":"Quantité S-1","Prio":"***","Danger":"INDISPO en test multi-step"},
    {"Feature":"qty_lag4","Cat":"Lag","Desc":"Quantité S-4","Prio":"***","Danger":"INDISPO après S+4"},
    {"Feature":"qty_lag26","Cat":"Lag","Desc":"Quantité S-26","Prio":"***","Danger":"NaN si historique court"},
    {"Feature":"rolling_mean_4w","Cat":"Rolling","Desc":"Moy mobile 4 sem","Prio":"****","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"rolling_mean_12w","Cat":"Rolling","Desc":"Moy mobile 12 sem","Prio":"****","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"rolling_mean_26w","Cat":"Rolling","Desc":"Moy mobile 26 sem","Prio":"***","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"rolling_std_12w","Cat":"Rolling","Desc":"Ecart-type 12 sem","Prio":"**","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"zero_rate_4w","Cat":"Zéros","Desc":"% zéros 4 sem","Prio":"****","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"zero_rate_12w","Cat":"Zéros","Desc":"% zéros 12 sem","Prio":"****","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"zero_rate_52w","Cat":"Zéros","Desc":"% zéros 52 sem","Prio":"***","Danger":"NaN si historique <52"},
    {"Feature":"pair_mean_qty","Cat":"Paire","Desc":"Moy historique paire","Prio":"*****","Danger":"Calculer sur données < S"},
    {"Feature":"pair_max_qty","Cat":"Paire","Desc":"Max historique paire","Prio":"***","Danger":"Calculer sur données < S"},
    {"Feature":"pair_nb_sem","Cat":"Paire","Desc":"Nb sem actives paire","Prio":"***","Danger":"Calculer sur données < S"},
    {"Feature":"agence_mean_qty","Cat":"Agence","Desc":"Moy ventes agence","Prio":"***","Danger":"Calculer sur données < S"},
    {"Feature":"article_mean_qty","Cat":"Article","Desc":"Moy ventes article","Prio":"***","Danger":"Calculer sur données < S"},
    {"Feature":"sem_sin","Cat":"Saison","Desc":"sin(2pi*sem/52)","Prio":"***","Danger":"OK"},
    {"Feature":"sem_cos","Cat":"Saison","Desc":"cos(2pi*sem/52)","Prio":"***","Danger":"OK"},
    {"Feature":"num_semaine","Cat":"Saison","Desc":"Numéro semaine brut","Prio":"**","Danger":"OK"},
    {"Feature":"region_agence","Cat":"Enrichi","Desc":"Région agence","Prio":"***","Danger":"OK (catégoriel)"},
    {"Feature":"famille_article","Cat":"Enrichi","Desc":"Famille/cat article","Prio":"****","Danger":"OK (catégoriel)"},
    {"Feature":"prix_unitaire","Cat":"Enrichi","Desc":"Prix facturation","Prio":"***","Danger":"Jointure à vérifier"},
    {"Feature":"trend_4w_vs_12w","Cat":"Tendance","Desc":"Ratio moy4/moy12","Prio":"***","Danger":"SHIFT(1) obligatoire"},
    {"Feature":"trend_yoy","Cat":"Tendance","Desc":"Ratio S/S-52","Prio":"***","Danger":"NaN si pas de N-1"},
])
line(df_to_str(features_table.sort_values("Prio", ascending=False), 50))
line(f"\n  LEGENDE :")
line(f"    SHIFT(1) obligatoire   = Data leakage si oublié")
line(f"    Calculer sur données<S = Ne pas inclure la semaine à prédire")
line(f"    INDISPO en test        = Le lag n'existe pas quand on prédit S+1 à S+26")
line(f"    OK                     = Safe, pas de risque")

# COMMAND ----------

# =====================================================================
# PRINT FINAL — COPIE-COLLE TOUT CE BLOC
# =====================================================================
section("FIN DU RAPPORT")

rapport = "\n".join(R)

# Sauvegarde DBFS
try:
    dbutils.fs.put("/FileStore/rapport_eda_sgdb.txt", rapport, overwrite=True)
    print("✅ Sauvegardé dans /FileStore/rapport_eda_sgdb.txt")
    print("   Téléchargement : https://<ton-workspace>.databricks.com/files/rapport_eda_sgdb.txt\n")
except:
    print("⚠️  dbutils.fs.put a échoué, mais le rapport est affiché ci-dessous.\n")

print("=" * 80)
print("  RAPPORT COMPLET — COPIE-COLLE TOUT CI-DESSOUS")
print("=" * 80)
print(rapport)